# UDFs (User Defined Functions)

## What is a UDF?
- UDF stands for **User Defined Function**.
- Allows us to apply custom Python logic on DataFrame columns.
- Used only when Spark does not provide a native function.

---

# Why Spark Runs on JVM

- Spark is written in **Scala**.
- Scala compiles into **JVM Bytecode**.
- JVM converts bytecode into machine code.
- Same Spark application runs on Windows, Linux, macOS without recompiling.

Advantages:
- Platform independent
- Automatic memory management (Garbage Collection)
- High performance

---

# What is Py4J?

PySpark runs two separate processes:

- JVM Process (Spark)
- Python Process (PySpark)

Both communicate using **Py4J**.

```
Python Process
      │
      │ Py4J
      ▼
JVM Process (Spark)
```

Most DataFrame operations send **only instructions** to JVM.

Example:

```python
df.filter(F.col("price") > 500)
```

Only the filter instruction is sent.

Data remains inside JVM.

---

# Why Python UDFs are Slow

For every row Spark performs:

1. Serialize JVM object
2. Send to Python
3. Execute Python function
4. Serialize result
5. Send back to JVM
6. Deserialize result

```
JVM
 ↓
Serialize
 ↓
Python
 ↓
Serialize
 ↓
JVM
```

For 100 million rows:

```
100 Million Rows
=
100 Million Boundary Crossings
```

Hence Python UDFs are much slower than native Spark functions.

---

# Catalyst Optimizer

Native Spark functions

- when()
- upper()
- regexp_replace()
- concat()
- round()

are optimized by Catalyst.

Python UDFs break Catalyst optimization because Spark cannot understand Python code.

---

# Creating a UDF

```python
from pyspark.sql.types import StringType

def categorize(price):
    if price > 500:
        return "Premium"
    return "Budget"

categorize_udf = F.udf(categorize, StringType())

df.withColumn(
    "category",
    categorize_udf(F.col("price"))
)
```

---

# Always Prefer Native Functions

Instead of

```python
categorize_udf(...)
```

Use

```python
F.when(F.col("price") > 500, "Premium")
 .otherwise("Budget")
```

Native functions are:
- Faster
- Optimized
- JVM only
- No serialization

---

# When Should We Use UDF?

Use a UDF only when:

- Complex business rules
- External Python libraries
- Custom algorithms
- Logic unavailable in Spark SQL functions

Otherwise always use native Spark functions.

---

# Pandas UDF

Regular UDF

```
1 Row
↓
Python
↓
1 Row
```

Pandas UDF

```
Entire Partition
↓
Apache Arrow
↓
Pandas Series
↓
Entire Partition
```

Instead of crossing the JVM-Python boundary **per row**, Pandas UDF transfers **one partition at a time**.

---

# Apache Arrow

Apache Arrow is an **in-memory columnar format** shared by JVM and Python.

Benefits:

- Very little serialization
- Faster data transfer
- Columnar batches
- Near zero-copy

---

# Regular UDF vs Pandas UDF

| Regular UDF | Pandas UDF |
|-------------|------------|
| Row-by-row | Partition-by-partition |
| Py4J | Apache Arrow |
| Slow | Much Faster |
| High Serialization | Low Serialization |

---

# Performance Hierarchy

```
Native Spark Function
        ↓
    Pandas UDF
        ↓
 Regular Python UDF
```

---


In [1]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-18")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-15ff433d-5ee0-45cb-9e9d-bbc39e42640d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 188ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.a

**Task 1**

Write a UDF called classify_order that takes unit_price and quantity as inputs and returns a string: "High Value" if revenue (price × quantity) > 1000, "Medium Value" if > 200, else "Low Value". Apply it to orders.csv.

In [2]:
%%time
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def classify_order(unit_price, quantity):
    revenue = unit_price * quantity

    if revenue > 1000:
        return "High Value"
    elif revenue > 200:
        return "Medium Value"
    else:
        return "Low Value"
classify_order_udf = F.udf(classify_order, StringType())
orders_df.withColumn(
    'revenue',
    F.col("unit_price")*F.col("quantity")
).select('revenue',
    classify_order_udf(F.col('unit_price'),F.col('quantity')).alias('revenue_status')
).show(5,truncate=False)

+-------+--------------+
|revenue|revenue_status|
+-------+--------------+
|2599.98|High Value    |
|449.99 |Medium Value  |
|1399.96|High Value    |
|179.98 |Low Value     |
|89.97  |Low Value     |
+-------+--------------+
only showing top 5 rows
CPU times: user 7.04 ms, sys: 11.8 ms, total: 18.8 ms
Wall time: 3.04 s


**Task 2**

Rewrite Task 1 using only native Spark functions (when() / otherwise()) instead of a UDF. Compare the two approaches — which is cleaner?

In [3]:
%%time
orders_df.withColumn(
    "revenue",
    F.col("quantity")*F.col('unit_price')
).select("revenue",
F.when(F.col('revenue')>1000,"High Value")
    .when(F.col("revenue")>200,'Medium Value')
    .otherwise("low_value").alias('revenue_status') ).show(5,truncate=False)

+-------+--------------+
|revenue|revenue_status|
+-------+--------------+
|2599.98|High Value    |
|449.99 |Medium Value  |
|1399.96|High Value    |
|179.98 |low_value     |
|89.97  |low_value     |
+-------+--------------+
only showing top 5 rows
CPU times: user 5.56 ms, sys: 8.81 ms, total: 14.4 ms
Wall time: 1.19 s


**Task 3**

Write a UDF that takes a customer's first_name and last_name and returns their initials — e.g. "James Anderson" → "J.A.". Apply it to customers.csv.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def initial_string(first_name, last_name):
    return first_name[0] + "." + last_name[0]

initial_string_udf = F.udf(initial_string, StringType())

customers_df.withColumn(
    "Name",
    initial_string_udf(F.col("first_name"), F.col("last_name"))
).show()

+-----------+-----------+---------+--------------------+-------------+-----+-------+-----------+----------+----+
|customer_id| first_name|last_name|               email|         city|state|country|signup_date|   segment|Name|
+-----------+-----------+---------+--------------------+-------------+-----+-------+-----------+----------+----+
|       C001|      James| Anderson|james.anderson@em...|     New York|   NY|    USA| 2021-03-15|Enterprise| J.A|
|       C002|      Maria|   Garcia|maria.garcia@emai...|  Los Angeles|   CA|    USA| 2021-05-22|       SMB| M.G|
|       C003|     Robert|  Johnson|robert.johnson@em...|      Chicago|   IL|    USA| 2020-11-08|Enterprise| R.J|
|       C004|      Linda| Martinez|linda.martinez@em...|      Houston|   TX|    USA| 2022-01-30|       SMB| L.M|
|       C005|    Michael|    Brown|michael.brown@ema...|      Phoenix|   AZ|    USA| 2021-07-19|   Startup| M.B|
|       C006|   Patricia|    Davis|patricia.davis@em...| Philadelphia|   PA|    USA| 2020-09-14|

**Task 4**

Try to rewrite Task 3 using only native Spark functions. Can you do it? What does this tell you about when a UDF is justified?

In [5]:
customers_df.withColumn(
    "Name",
    F.concat_ws(
        ".",
        F.substring("first_name", 1, 1),
        F.substring("last_name", 1, 1)
    )
).show()

+-----------+-----------+---------+--------------------+-------------+-----+-------+-----------+----------+----+
|customer_id| first_name|last_name|               email|         city|state|country|signup_date|   segment|Name|
+-----------+-----------+---------+--------------------+-------------+-----+-------+-----------+----------+----+
|       C001|      James| Anderson|james.anderson@em...|     New York|   NY|    USA| 2021-03-15|Enterprise| J.A|
|       C002|      Maria|   Garcia|maria.garcia@emai...|  Los Angeles|   CA|    USA| 2021-05-22|       SMB| M.G|
|       C003|     Robert|  Johnson|robert.johnson@em...|      Chicago|   IL|    USA| 2020-11-08|Enterprise| R.J|
|       C004|      Linda| Martinez|linda.martinez@em...|      Houston|   TX|    USA| 2022-01-30|       SMB| L.M|
|       C005|    Michael|    Brown|michael.brown@ema...|      Phoenix|   AZ|    USA| 2021-07-19|   Startup| M.B|
|       C006|   Patricia|    Davis|patricia.davis@em...| Philadelphia|   PA|    USA| 2020-09-14|

This task shows that UDFs should only be used when Spark's built-in functions cannot solve the problem.

Why prefer native Spark functions?

✅ Faster because Spark can optimize them (Catalyst Optimizer).

✅ Run entirely in the JVM, avoiding Python-JVM serialization overhead.

✅ Support code generation (Tungsten), making execution more efficient.


✅ Easier for Spark to optimize and parallelize.

Python UDFs:

❌ Slower due to data serialization between the JVM and Python.

❌ Prevent many Spark query optimizations.

❌ Should be the last option.


Rule of thumb
Use native Spark functions (concat_ws, substring, when, regexp_replace, coalesce, etc.) whenever possible.
Use a UDF only when your business logic cannot be expressed using Spark's built-in functions.

For this task, a UDF is not justified because Spark already provides substring() and concat_ws() to achieve the same result more efficiently.

In [6]:
spark.stop()